# Code as Policies with CaP-X

[CaP-X](https://github.com/capgym/cap-x) (*Code-as-Policies eXtended*) is a framework for benchmarking and improving coding agents for robot manipulation. Instead of asking a model what the arm should do next, it asks the model to **write a Python program** that solves the task by composing perception and control primitives, runs that program in simulation, and scores whether the task actually got done.

The robot is a Franka Panda again, this time in a Robosuite/MuJoCo simulation. The model that writes the program is served locally, so the reasoning stays on your Radeon GPU.

> **Current workshop image:** live perception uses ungated OWLv2 + SAM2 with image-baked weights. Some saved outputs in this notebook were captured before that migration and mention SAM3; rerunning those cells replaces the historical logs.

## Goals

* Understand how a code-generating agent controls a robot, and how that differs from the tool-calling agent in the previous notebook
* Configure CaP-X against your own OpenAI-compatible model server
* Read the program the model writes, and the prompt that produced it
* Measure a model by its success rate over repeated trials instead of a single demo

## Two ways to put an LLM on a robot

Notebook 03 and this notebook attempt to solve the same problem of robot manipulation through using natural language as an input, and planned motion of the arm as an output. Their approach differs with the local model's responsibility: **calling tools vs generating code.**

| | Notebook 03 - RAI | This notebook - CaP-X |
| --- | --- | --- |
| What the model emits | one tool call at a time | one Python program, up front |
| Where the model sits | inside the control loop | upstream of it |
| Model calls per task | one per step, dozens of them | one |
| What drives the robot | a LangGraph loop | CPython executing the generated code |
| Perception | a ROS 2 service, invoked as a tool | OWLv2, SAM2, and Contact-GraspNet, called as functions from inside the program |
| How you judge it | watch the arm | reward over N seeded trials, i.e. a success rate |
| A failure looks like | a tool call you can read | a traceback, or a program that runs cleanly and still misses |

The main tradeoff is: adaptability against efficiency. RAI can attempt any task with no preparation, but the model has to reason from scratch every time, so the hundredth run of a task costs the same as the first. CaP-X generates a program per task, and that program can then be re-run without a model at all; yet a wrong read of the scene spoils the whole episode, and every new task needs another program.

## Serve the model locally

CaP-X needs an OpenAI-compatible chat-completions endpoint. We use the same Lemonade server and port as notebooks 02 and 03.

`ensure_lemonade` starts the daemon if needed and loads the image-cached model. It reports elapsed setup time so we can separate one-time startup from inference and rollout.

In [1]:
import sys
from time import perf_counter

sys.path.insert(0, "/ryzers")

from capx_demo import (
    benchmark,
    ensure_lemonade,
    llama_metrics,
    metric_delta,
    quiet_output,
    show_trial_grid,
    show_video,
)

MODEL = "Gemma-4-E2B-it-GGUF"
LEMONADE_SECONDS = ensure_lemonade(MODEL)

Loading Gemma-4-E2B-it-GGUF from the image cache...


Lemonade ready in 4.5s


## Start the perception services and configure CaP-X

Four servers sit behind the primitives the generated program may call. Ungated **OWLv2** grounds a noun phrase such as `"red cube"` into a bounding box, ungated **SAM2** turns that box into a pixel mask, Contact-GraspNet turns the mask plus depth into ranked 6-DoF grasp poses, and PyRoKi solves inverse kinematics. Their weights are baked into the course image, so this step needs neither a Hugging Face token nor a model download. `FrankaControlApi` wraps them into the five functions the model is allowed to use.

The cell below is what `capx/envs/launch.py` does before its first trial, one call at a time. `LaunchArgs` is the dataclass the CLI parses its flags into, so filling one in by hand configures the framework exactly as a terminal run would, and `model` and `server_url` are the whole coupling between CaP-X and whatever is doing the reasoning. Point them at any OpenAI-compatible chat-completions endpoint and nothing else in this notebook changes.

In [2]:
SERVER_URL = "http://localhost:13305/api/v1/chat/completions"
CONFIG_PATH = "env_configs/cube_stack/franka_robosuite_cube_stack.yaml"
TEMPERATURE = 0.2
MAX_TOKENS = 16384

service_started = perf_counter()
with quiet_output() as service_log:
    from capx.envs.configs.instantiate import instantiate
    from capx.envs.launch import LaunchArgs
    from capx.envs.runner import _start_api_servers
    from capx.utils.launch_utils import _load_config

    args = LaunchArgs(
        config_path=CONFIG_PATH,
        model=MODEL,
        server_url=SERVER_URL,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    env_factory, config, api_servers = _load_config(args)
    servers = _start_api_servers(api_servers, 900.0)
    env = instantiate(env_factory)
    api = next(iter(env._apis.values()))
    obs, _ = env.reset(options={"trial": 0}, seed=0)

SERVICE_SECONDS = perf_counter() - service_started
print(
    f"Ready in {SERVICE_SECONDS:.1f}s: {type(env).__name__} + "
    f"{type(api).__name__} (details: {service_log})"
)

Ready in 11.4s: FrankaPickPlaceCodeEnv + FrankaControlApi (details: /tmp/capx-services.log)


## Generate a program for the task

`ModelQueryArgs` carries the same values as `LaunchArgs`, and `query_model` is the chat-completions call underneath, so this is one HTTP round trip to the server you configured above. `obs["full_prompt"]` is the two-message conversation the environment built for the scene it reset to.

In [3]:
from capx.llm.client import ModelQueryArgs, query_model
from capx.utils.launch_utils import _extract_code

query_args = ModelQueryArgs(
    model=MODEL,
    server_url=SERVER_URL,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

metrics_before = llama_metrics()
query_started = perf_counter()
with quiet_output():
    response = query_model(query_args, obs["full_prompt"])
LLM_WALL_SECONDS = perf_counter() - query_started
LLM_METRICS = metric_delta(metrics_before, llama_metrics())

blocks = _extract_code(response["content"])
assert blocks, f"no code in the reply - raise MAX_TOKENS?\n{response['content'][-500:]}"
program = blocks[0]

print(
    f"LLM: {LLM_WALL_SECONDS:.1f}s wall · "
    f"{LLM_METRICS.get('prompt_tokens_total', 0):.0f} prompt tokens · "
    f"{LLM_METRICS.get('tokens_predicted_total', 0):.0f} generated tokens"
)
print(program)

LLM: 14.2s wall · 875 prompt tokens · 1401 generated tokens
red_grasp_pos, red_grasp_quat = sample_grasp_pose("red cube")

# 1. Grasp the red cube
# Move to the grasp pose with a 0.1m approach
goto_pose(red_grasp_pos, red_grasp_quat, z_approach=0.1)
close_gripper()

# 2. Lift the cube to a safe height (at least +0.2m in Z)
lift_z = 0.2
lifted_pos = (red_grasp_pos[0], red_grasp_pos[1], red_grasp_pos[2] + lift_z)
goto_pose(lifted_pos, red_grasp_quat, z_approach=0.1)

# 3. Determine placement position on the green cube
green_pos, _, green_extent = get_object_pose("green cube", return_bbox_extent=True)

# Calculate the target Z height for placement
# place_z = green_center_z + green_extent[2]/2 + red_extent[2]/2
place_z = green_pos[2] + (green_extent[2] / 2.0) + (red_extent[2] / 2.0)

# The X and Y position should align with the center of the green cube for stacking
placement_pos = (green_pos[0], green_pos[1], place_z)

# 4. Move to the placement position
# Reuse the grasp orientation for 

## Run program

**The generated code is the policy.**

If it fails, the output will display how. `No detections` means the noun phrase did not ground, a traceback means the model got Python wrong (the most common failure for small models), and a clean run below `1.0` means the geometry was wrong.

In [4]:
env.enable_video_capture(True, clear=True)

rollout_started = perf_counter()
with quiet_output():
    _, reward, terminated, _, info = env.step(program)
ROLLOUT_SECONDS = perf_counter() - rollout_started

print(
    f"Rollout: {ROLLOUT_SECONDS:.1f}s · reward {reward:.3f} · "
    f"solved {info['task_completed']}"
)
if info["sandbox_rc"]:
    print("\nTraceback (tail):\n" + info["stderr"][-1200:])

show_video(env)

Rollout: 11.8s · reward 0.004 · solved False

Traceback (tail):
Traceback (most recent call last):
  File "/ryzers/cap-x/capx/envs/tasks/base.py", line 174, in _exec_user_code
    exec(code, self._exec_globals, self._exec_globals)
  File "<string>", line 18, in <module>
NameError: name 'red_extent' is not defined. Did you mean: 'green_extent'?

Saved interaction video to /tmp/capx_notebook/video_notebook_run.mp4 (87 frames)


'/tmp/capx_notebook/video_notebook_run.mp4'

## Results over many layouts

A single trial does not measure much. Each trial reseeds the cube positions, so a program that succeeds on one arrangement can fail on the next, and only a success rate over several trials is comparable between models.

The cell below runs `launch.py` five times and prints the reward for each trial, followed by the success rate and the mean reward. Each trial also writes its own directory holding the generated program, the model's raw response, the prompt and an MP4 of the episode.

Pass `oracle=True` to run the reference program instead of calling the model.

In [5]:
benchmark_metrics_before = llama_metrics()
benchmark_started = perf_counter()
trials = benchmark(
    model=MODEL,
    server_url=SERVER_URL,
    config_path=CONFIG_PATH,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    trials=5,
)
BENCHMARK_SECONDS = perf_counter() - benchmark_started
BENCHMARK_METRICS = metric_delta(benchmark_metrics_before, llama_metrics())

print(
    f"Five trials: {BENCHMARK_SECONDS:.1f}s wall · "
    f"{BENCHMARK_METRICS.get('tokens_predicted_seconds_total', 0):.1f}s "
    "inside LLM generation"
)
show_trial_grid(trials)

Running 5 CaP-X trials...


CaP-X finished in 141.6s
trial  sandbox  reward  solved
    1       ok   0.004   False
    2    error   0.003   False
    3       ok   0.002   False
    4       ok   0.001   False
    5       ok   1.000    True

success rate: 1/5   mean reward: 0.202
Five trials: 141.6s wall · 77.7s inside LLM generation


5 rollout videos · 0.1 MB embedded


In [6]:
print("Timing breakdown")
print(f"  Lemonade/model setup: {LEMONADE_SECONDS:6.1f}s  (one time)")
print(f"  perception/control setup: {SERVICE_SECONDS:6.1f}s  (one time)")
print(f"  one LLM generation: {LLM_WALL_SECONDS:6.1f}s")
print(f"  one simulator rollout: {ROLLOUT_SECONDS:6.1f}s")
print(f"  five complete trials: {BENCHMARK_SECONDS:6.1f}s")

Timing breakdown
  Lemonade/model setup:    4.5s  (one time)
  perception/control setup:   11.4s  (one time)
  one LLM generation:   14.2s
  one simulator rollout:   11.8s
  five complete trials:  141.6s


### Why this notebook uses cube stacking

A live five-trial sweep with the same Gemma E2B model and settings tested all four single-arm workshop tasks. Cube stacking was the only task with a completed generated policy (**1/5**); cube lifting, cube restacking, and spill wiping each completed **0/5**. The result is stochastic, so a rerun can differ, but cube stacking currently gives the clearest teaching example: the videos above include a real successful placement while the failures still expose useful code and motion-planning mistakes.

## Where the motion happens

`goto_pose` is the one primitive that moves the arm, and it does three things worth knowing about. Every pose in this API is a gripper-tip pose, so it adds the 10.7 cm offset to the `panda_hand` link before solving, or the arm drives into the table. A non-zero `z_approach` makes it solve and execute *twice*, once at a standoff along the gripper's local -Z and once at the target, which is why the task prompt insists on `z_approach=0.1`. And it hands the previous joint configuration back to the solver, which then penalises joint velocity, so the elbow does not flip across the table between waypoints.

The solve is nonlinear least squares over the joint vector, so it always returns something: an unreachable target comes back as the closest configuration it could find, with no error and no flag. When a program fails for no visible reason, suspect that first.

In [8]:
import inspect

print(inspect.getsource(type(api).goto_pose))

    def goto_pose(
        self, position: np.ndarray, quaternion_wxyz: np.ndarray, z_approach: float = 0.0
    ) -> None:
        """Go to pose using Inverse Kinematics.
        There is no need to call a second goto_pose with the same position and quaternion_wxyz after calling it with z_approach.
        Args:
            position: (3,) XYZ in meters.
            quaternion_wxyz: (4,) WXYZ unit quaternion.
            z_approach: (float) Z-axis distance offset for goto_pose insertion approach motion. Will first arrive at position + z_approach meters in Z-axis before moving to the requested pose. Useful for more precise grasp approaches. Default is 0.0.
        Returns:
            None
        """
        pos_str = np.array2string(np.asarray(position), precision=4)
        approach_info = f" (z_approach={z_approach:.3f})" if z_approach != 0.0 else ""
        self._log_step("goto_pose", f"Moving to position {pos_str}{approach_info} …")

        pos = np.asarray(position, dtype=np.floa

## System Prompt

Two messages are used as prompts for the api calls, the ones handed to `query_model` above. The system message is one sentence, and the user message is the task description followed by an `APIs:` section that is not written by hand anywhere: CaP-X renders it from the signature and docstring of every primitive the program may call. So the docstrings *are* the prompt, and it cannot drift out of date with the code.

In [9]:
system, user = obs["full_prompt"]

print(system["content"])
print(user["content"][0]["text"])

You are a helpful assistant that generates Python code to directly solve the task.

You are controlling a Franka Emika robot with the API described below.
Goal: Pick up the red cube and gently stack it on top of the green cube, then release it.

Key rules:
- The extent from get_object_pose(..., return_bbox_extent=True) is the FULL side length. Use extent[2]/2 for half-height.
- For placement orientation, reuse the grasp quaternion from sample_grasp_pose. Do NOT use the quaternion from get_object_pose (it is unreliable for orientation).
- Always use z_approach=0.1 when approaching an object for grasping or placing.
- After grasping, lift the cube to a safe height (at least +0.2m in Z) before moving laterally to the placement location.
- The stacking height formula is: place_z = green_center_z + green_extent[2]/2 + red_extent[2]/2
- Nothing should be dropped from a height. Always approach with z_approach for controlled descent.

Write ONLY executable Python code (no code fences). Import 

> **Note:** nothing in that prompt describes the scene. There is no image, no object coordinates and no joint state, so the model writes the program blind, and every fact about the actual arrangement is obtained at runtime by the program itself, through the perception servers.

## Key Takeaways

Now you know:
- How a Code-as-Policies agent differs from a tool-calling agent: one program written up front instead of one decision per step, and a success rate instead of a demo
- How to configure CaP-X against any OpenAI-compatible server, with `LaunchArgs` and `ModelQueryArgs` carrying the same `--model` and `--server-url` the CLI takes
- How the generated program gets from a noun phrase to motion: OWLv2 grounding into a box, SAM2 segmentation into a mask, mask plus depth into a grasp pose, pose into joint angles through IK
- That the prompt is generated from the primitives' docstrings, so the API and its documentation cannot drift apart
- How to benchmark a locally served model and read the result

## What to Try Next

- Point `SERVER_URL` at a llama.cpp, Ollama or vLLM server, or at the OpenRouter proxy, and rerun from that cell down to understand how different models perform and how to use different backends
- Run `benchmark(..., oracle=True)` to see the stack succeed with no model in the loop, and use it as your control whenever a result looks wrong
- Edit a docstring in `FrankaControlApi` and watch the generated program change; you are editing the prompt. `/ryzers/cap-x` is an editable install, so the change lands on the next kernel restart
- Raise `temperature` and generate a few programs for the same scene, to see how much of the policy the model is guessing

## References

* [CaP-X](https://github.com/capgym/cap-x)
* [OWLv2 Large](https://huggingface.co/google/owlv2-large-patch14-ensemble) · [SAM2.1 Large](https://huggingface.co/facebook/sam2.1-hiera-large) · [Contact-GraspNet](https://github.com/NVlabs/contact_graspnet) · [PyRoKi](https://github.com/chungmin99/pyroki)
* Optional fast profile: [OWLv2 Base](https://huggingface.co/google/owlv2-base-patch16-ensemble) · [SAM2.1 Small](https://huggingface.co/facebook/sam2.1-hiera-small)
* [Robosuite](https://github.com/ARISE-Initiative/robosuite)
* [Lemonade](https://lemonade-server.ai/)